In [17]:
# import important libraries

In [18]:
import pandas as pd
import numpy as np

In [2]:
import nltk
import re

In [3]:
from pathlib import Path

In [4]:
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.stem import WordNetLemmatizer

In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer

In [58]:
from sklearn.naive_bayes import MultinomialNB

In [9]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.ensemble import RandomForestClassifier 

In [ ]:
# data loading

In [12]:
df = pd.read_csv('all_kindle_review .csv')
df.head()

,Unnamed: 0.1,Unnamed: 0,asin,helpful,rating,reviewText,reviewTime,reviewerID,reviewerName,summary,unixReviewTime
0,0,11539,B0033UV8HI,"[8, 10]",3,"Jace Rankin may be short, but he's nothing to ...","09 2, 2010",A3HHXRELK8BHQG,Ridley,Entertaining But Average,1283385600
1,1,5957,B002HJV4DE,"[1, 1]",5,Great short read. I didn't want to put it dow...,"10 8, 2013",A2RGNZ0TRF578I,Holly Butler,Terrific menage scenes!,1381190400
2,2,9146,B002ZG96I4,"[0, 0]",3,I'll start by saying this is the first of four...,"04 11, 2014",A3S0H2HV6U1I7F,Merissa,Snapdragon Alley,1397174400
3,3,7038,B002QHWOEU,"[1, 3]",3,Aggie is Angela Lansbury who carries pocketboo...,"07 5, 2014",AC4OQW3GZ919J,Cleargrace,very light murder cozy,1404518400
4,4,1776,B001A06VJ8,"[0, 1]",4,I did not expect this type of book to be in li...,"12 31, 2012",A3C9V987IQHOQD,Rjostler,Book,1356912000


In [13]:
df = df[['reviewText', 'rating']]
df.sample(5)

,reviewText,rating
2640,These stories kept my attention and were hard ...,4
1347,PLOT: Lila's father is going to lose his busin...,5
10827,This book had a really messed up female charac...,3
3223,"Third book in ""Triple Trouble"" series, precede...",3
3153,I think there is no better story than that of ...,5


In [14]:
df.rating.value_counts()

rating
5    3000
4    3000
3    2000
2    2000
1    2000
Name: count, dtype: int64

In [15]:
# convereting 1,2,3 to 0 -> negative
# converting 4,5 to 1 -> positive

df.rating = df.rating.replace([1,2,3],0)
df.rating = df.rating.replace([4,5],1)

df['sentiment'] = df['rating']
df.drop(columns=['rating'], inplace=True)

df.head()

,reviewText,sentiment
0,"Jace Rankin may be short, but he's nothing to ...",0
1,Great short read. I didn't want to put it dow...,1
2,I'll start by saying this is the first of four...,0
3,Aggie is Angela Lansbury who carries pocketboo...,0
4,I did not expect this type of book to be in li...,1


In [16]:
# basic analysis of the data

# shape of the DataFrame
print("Dataset shae:", df.shape)

#number of positive vs neagtive reviews
label_counts = df['sentiment'].value_counts()
print("\nCount of each sentiment:\n", label_counts)

#proportion of positive vs neagtive reviews
print("\nProportion of each sentiment:\n", label_counts / len(df))

#check for missing values
missing_values = df.isnull().sum()
print("\nMissing values per column:\n", missing_values)

# average review length by sentiment
df['reviewText_length']= df['reviewText'].apply(len)
avg_length = df.groupby('sentiment')['reviewText_length'].mean()
print("/nAverage review length by sentiment:\n", avg_length)

#number of unique reviews
num_unique_reviews = df['reviewText'].nunique()
print("\nNumber of unique reviews:", num_unique_reviews)

#show some sample positive reviews
print("\nSample positive reviews:")
print(df[df['sentiment'] == 1]['reviewText'].head())

#show some sample negative reviews
print("\nSample negative reviews:")
print(df[df['sentiment'] == 0]['reviewText'].head())

Dataset shae: (12000, 2)

Count of each sentiment:
 sentiment
0    6000
1    6000
Name: count, dtype: int64

Proportion of each sentiment:
 sentiment
0    0.5
1    0.5
Name: count, dtype: float64

Missing values per column:
 reviewText    0
sentiment     0
dtype: int64
/nAverage review length by sentiment:
 sentiment
0    605.118667
1    598.006833
Name: reviewText_length, dtype: float64

Number of unique reviews: 12000

Sample positive reviews:
1    Great short read.  I didn't want to put it dow...
4    I did not expect this type of book to be in li...
5    Aislinn is a little girl with big dreams. Afte...
7    I got this because I like collaborated short s...
8    Loved this book, I am hooked on this series an...
Name: reviewText, dtype: object

Sample negative reviews:
0     Jace Rankin may be short, but he's nothing to ...
2     I'll start by saying this is the first of four...
3     Aggie is Angela Lansbury who carries pocketboo...
6     This has the makings of a good story... unf

In [19]:
# data cleaning and preprocessing

In [20]:
# check for null values
df.isnull().sum()

reviewText           0
sentiment            0
reviewText_length    0
dtype: int64

In [21]:
df.sentiment.value_counts()

sentiment
0    6000
1    6000
Name: count, dtype: int64

In [22]:
# removing special char
df['reviewText'] = df['reviewText'].apply(lambda x: re.sub('[^a-zA-Z0-9 ]+', '', str(x)))
# remove URLs from reviewText
df['reviewText'] = df['reviewText'].apply(lambda x: re.sub(r'http\S+|www.\S+', '', x))
# remove aditional spaces
df['reviewText']=df['reviewText'].apply(lambda x:" ".join(x.split()))

In [23]:
lemmatizer = WordNetLemmatizer()

def preprocess_text(text: str) -> str:
    """clean, lowercase, and lemmatize tokens."""
    tokens = text.lower().split()
    filtered = [lemmatizer.lemmatize(token) for token in tokens]
    return ''.join(filtered)

def tokenize_text(text: str) -> list[str]:
    """ Return tokenized version of the cleaned text."""
    return preprocess_text(text).split()

def build_corpus(texts) -> list[str]:
    """build a corpus of cleaned messages."""
    return [preprocess_text(text) for text in texts]


In [24]:
corpus = build_corpus(df['reviewText'])
labels = df['sentiment']

# tokenized corpus for Word2Vec (list of tokens per document)
tokenized_corpus = [text.split() for text in corpus]

preview = pd.DataFrame({'original': df['reviewText'].head(5), 'cleaned':corpus[:5]})
preview

,original,cleaned
0,Jace Rankin may be short but hes nothing to me...,jacerankinmaybeshortbuthenothingtomesswithathe...
1,Great short read I didnt want to put it down s...,greatshortreadididntwanttoputitdownsoireadital...
2,Ill start by saying this is the first of four ...,illstartbysayingthisisthefirstoffourbooksoiwas...
3,Aggie is Angela Lansbury who carries pocketboo...,aggieisangelalansburywhocarrypocketbookinstead...
4,I did not expect this type of book to be in li...,ididnotexpectthistypeofbooktobeinlibrarywaplea...


In [25]:
print(corpus[0])
print(labels[0])

jacerankinmaybeshortbuthenothingtomesswithathemanwhowajusthauledoutofthesaloonbytheundertakerknownowheafamousbountyhunterinoregoninthe1890swhowhenheshotthemaninthesaloonjustfinishedayearlongquesttoavengehissistermurderandisnowtryingtofigureoutwhattodonextwhenthesnottynosedfarmboyhejustrescuedfromagangofbullyofferhimmoneytokillamanwhoforcedhimoffhisranchhereluctantlyagreestobringthemantojusticebutnottokillhimoutrightbutfirstheneedtotellhissisterwidowerthenewskylakylespringerbaileyhabeenridingthetrailandsleepingonthegroundforthepastmonthwhiletryingtofindjaceshewantrevengeonthemanwhokilledherhusbandandtookherranchamongstothercrimeandshesnotsokeenonthedetourjacewanttotakebutsherealizesshesoutofoptionsoshehidebehindherboypersonaabestshecanandtrytokeeppacewhenaconfrontationalongthewaygethershotandjacediscoversthatkylesakylashehatocomecleanaboutthewholereasonsheneedthisscoundreldeadandhopehellstillhelpherthebookhaitshareoftouchingmomentandslowbloomingromancekylawefindouthagoodreasontofearmena

In [26]:
# trainTestSplit

In [30]:
def split_text(corpus,labels,test_size: float = 0.2,random_state=120):
    """Split the data into training and test sets."""
    return train_test_split(
        corpus, labels, test_size=test_size, random_state=random_state, stratify=labels
    )

In [31]:
# Train/test split
X_train_text, X_test_text, y_train, y_test = split_text(
    corpus,labels,test_size=0.2,random_state=120)

In [32]:
# analyze the train/test split: print size, shape, label counts, and distribution

print("Train/test split analysis:")
print(f"Number of training sample: {len(X_train_text)}")
print(f"Number of test sample: {len(X_test_text)}\n")

Train/test split analysis:
Number of training sample: 9600
Number of test sample: 2400



In [33]:
# show a few examples from each set
print("Sample training messages:")
print(pd.Series(X_train_text).head(3))
print(pd.Series(X_test_text).head(3))
print()

Sample training messages:
0    thisbookreadliketheworkofatalentedteenageritla...
1    ilikeviviwishshehadavarietyifwomaninherbooknot...
2    thiswanotanythingspecialithoughtitmightbeafunr...
dtype: object
0    iwafrustratedwiththisbookireallylovedohnersnew...
1    ireadandenjoyparanormalfantasyromancetheheroin...
2    samanthacrosspromisedhermotherthatshewouldalwa...
dtype: object



In [34]:
# analysis label distribution
y_train_counts = pd.Series(y_train).value_counts().sort_index()
y_test_counts = pd.Series(y_test).value_counts().sort_index()
print("Label distribution in training set:")
print(y_train_counts)
print("Label distribution in test set:")
print(y_test_counts)
print()

Label distribution in training set:
sentiment
0    4800
1    4800
Name: count, dtype: int64
Label distribution in test set:
sentiment
0    1200
1    1200
Name: count, dtype: int64



In [35]:
# Percentage calculation
train_pct = (y_train_counts / len(y_train) * 100).round(2)
test_pct = (y_test_counts / len(y_test) * 100).round(2)
print("Label percentage in training set:")
print(train_pct)
print("Label percentage in test set:")
print(test_pct)
print()

Label percentage in training set:
sentiment
0    50.0
1    50.0
Name: count, dtype: float64
Label percentage in test set:
sentiment
0    50.0
1    50.0
Name: count, dtype: float64



In [36]:
# feature extraction

In [39]:
! pip install gensim

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/24.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/24.4 MB ? eta -:--:--
   ---------------------------------------- 0.3/24.4 MB ? eta -:--:--
    --------------------------------------- 0.5/24.4 MB 1.2 MB/s eta 0:00:21
    --------------------------------------- 0.5/24.4 MB 1.2 MB/s eta 0:00:21
   - -------------------------------------- 0.8/24.4 MB 1.0 MB/s eta 0:00:24
   -- ------------------------------------- 1.3/24.4 MB 1.2 MB/s eta 0:00:19
   -- ------------------------------------- 1.6/24.4 MB 1.4 MB/s eta 0:00:17
   --- ------------------------------------ 2.1/24.4 MB 1.5 MB/s eta 0:00:15
   --- ------------------------------------ 2.4/24.4 MB 1.5 MB/s eta 0:00:15
   ---- ----------------------------------- 2.9/24.4 MB 1.6 MB/s eta 0:00:14
   ----- ---------------------------------- 3.4/24.4 MB 1.7 MB/s eta 0:00:13
   ------ ---------

In [41]:
import gensim
import gensim.downloader as api

In [45]:
# using pre-trained google news word2vec instead of training on our corpus
def load_google_word2vec(model_name: str = "word2vec-google-news-300") -> gensim.models.KeyedVectors:
    """ load google's pre-trained word2vec mpdel(cached after first download)"""
    return api.load(model_name)
    

In [79]:
def average_word_vectors(docs, vectors: gensim.models.KeyedVectors) -> np.ndarray:
    """Average pre-trained word vectors for each document; returns 2D numpy array."""
    doc_vectors = []
    for doc in docs:
        tokens = doc.split() if isinstance(doc, str) else list(doc)
        token_vectors = [vectors[token] for token in tokens if token in vectors.key_to_index]
        if not token_vectors:
            # If no known words, add a zero-vector of the correct size
            token_vectors = [np.zeros(vectors.vector_size)]
        doc_vectors.append(np.mean(token_vectors, axis=0))
    return np.vstack(doc_vectors)

In [80]:
google_word2vec = load_google_word2vec()

In [81]:
# toenize train\test splits for Word2Vec averaging
X_train_tokens = [text.split() for text in X_train_text]
X_test_tokens = [text.split() for text in X_test_text]

X_train_word2vec = average_word_vectors(X_train_tokens, google_word2vec)
X_test_word2vec = average_word_vectors(X_test_tokens, google_word2vec)

In [82]:
X_train_word2vec = average_word_vectors(X_train_tokens, google_word2vec)
X_test_word2vec = average_word_vectors(X_test_tokens, google_word2vec)

In [83]:
print("Word2Vec feature matrix shape (train):", X_train_word2vec.shape)
print("Word2Vec feature matrix shape (test):", X_test_word2vec.shape)
print("Embedding dimensionality:", google_word2vec.vector_size)

Word2Vec feature matrix shape (train): (9600, 300)
Word2Vec feature matrix shape (test): (2400, 300)
Embedding dimensionality: 300


In [84]:
# model training and evaluation

In [85]:
model = RandomForestClassifier(
    n_estimators=500,max_depth=10,min_samples_split=2,min_samples_leaf=1
)

In [86]:
def train_model(x_train, y_train):
    model.fit(x_train, y_train)
    return model

In [87]:
def evaluate_model(model, X, y):
    predictions = model.predict(X)
    accuracy = accuracy_score(y, predictions)
    report = classification_report(y, predictions, target_names=['negative', 'positive'])
    return accuracy, report

In [88]:
model = train_model(X_train_word2vec, y_train)

In [89]:
accuracy, report = evaluate_model(model, X_test_word2vec, y_test)

C:\ProgramData\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\ProgramData\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\ProgramData\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [90]:
print(f'Accuracy: {accuracy:.3f}')
print(report)

Accuracy: 0.500
              precision    recall  f1-score   support

    negative       0.50      1.00      0.67      1200
    positive       0.00      0.00      0.00      1200

    accuracy                           0.50      2400
   macro avg       0.25      0.50      0.33      2400
weighted avg       0.25      0.50      0.33      2400

